# smoke-cloud-example — generated notebook (template v1.0.0); DO NOT EDIT.
#
# Produced by `spectraquant cloud notebook` from a frozen RunSpec. Hand edits are overwritten on the
# next regeneration and are not accepted as evidence (AGENTS.md §2b rule 2).
#
# platform: colab | gpu_required: True | timeout: 720 min
# experiment: configs/experiment/smoke.yaml (none)
# git_commit: 5784d133d3a5855739b58c7421f6460fdd7e9ccf (DIRTY — not reproducible)
# measurement_class_expected: 1
# spec_sha256: sha256:b9cab90dfc2d76b5573b90b504d832b2843bda042400edb76fda33c1976e0d46
# notebook_digest: sha256:cedc67828042c93a136c0f8d44aea37d5bcba7c0242586421238655bc82b03f5
#
# Cells, in order: environment, install, repo, data, run, manifest, export, teardown.


In [ ]:
# Mandatory cell 1/8 — environment record.
# Nothing scientific happens here: hardware, dependency versions, start time and the notebook's own
# identity are recorded so the manifest can prove where the numbers came from.
import hashlib, json, os, platform as _platform, shutil, subprocess, sys, time
from pathlib import Path

SPEC = json.loads('{"allow_dirty":true,"dataset_refs":[{"checksum":"sha256:f6cc7b86b273cfb2c2e07b68cb33a6229fab6871ef8e72aa6ed912611d9d965a","name":"synthetic-lcg-v1","revision":"config-sha256:c2b1bec28f410e089a848327a1ae8deda41ad8c850a1dc0abf2797be9bf45677","split":"train"}],"expected_artifacts":[{"min_bytes":64,"name":"run_manifest.json","sha256":null},{"min_bytes":64,"name":"artifacts/sample-results/smoke-manifest.json","sha256":null}],"experiment_config":"configs/experiment/smoke.yaml","git_commit":"5784d133d3a5855739b58c7421f6460fdd7e9ccf","gpu_required":true,"install_spec":"uv sync --frozen --extra cloud","max_cost_authorized_usd":null,"measurement_class_expected":1,"overrides":[],"platform":"colab","python_version":"3.11","repo_url":"https://github.com/OussemaHarrabi/SpectraQuant.git","run_id":"smoke-cloud-example","seeds":[1234],"timeout_minutes":720}')
RUN_ID = SPEC["run_id"]
SPEC_SHA256 = 'sha256:b9cab90dfc2d76b5573b90b504d832b2843bda042400edb76fda33c1976e0d46'
NOTEBOOK_DIGEST = 'sha256:cedc67828042c93a136c0f8d44aea37d5bcba7c0242586421238655bc82b03f5'
NOTEBOOK_TEMPLATE_VERSION = '1.0.0'

WORKDIR = Path(os.environ.get("SPECTRAQUANT_WORKDIR") or (Path.cwd() / "spectraquant-work" / RUN_ID))
REPO_DIR = WORKDIR / "repo"
ARTIFACT_DIR = WORKDIR / "artifacts" / "run"
LOG_DIR = WORKDIR / "logs"
EXPORT_DIR = Path(os.environ.get("SPECTRAQUANT_EXPORT_DIR") or (Path.cwd() / "spectraquant-export" / RUN_ID))
for _directory in (WORKDIR, ARTIFACT_DIR, LOG_DIR, EXPORT_DIR):
    _directory.mkdir(parents=True, exist_ok=True)


def _capture(argv, *, cwd=None):
    """Run a command and return (returncode, combined stdout+stderr); never raises."""
    try:
        proc = subprocess.run(
            [str(item) for item in argv],
            cwd=None if cwd is None else str(cwd),
            capture_output=True,
            text=True,
        )
    except (OSError, subprocess.SubprocessError) as exc:
        return 127, f"{type(exc).__name__}: {exc}"
    return proc.returncode, (proc.stdout or "") + (proc.stderr or "")


def _require_ok(rc, out, what):
    """Abort the notebook when a structural step fails (a truncated run must be obvious)."""
    if rc != 0:
        raise RuntimeError(f"{what} failed (exit {rc}):\n{out[-4000:]}")


def _ram_bytes():
    try:
        import psutil  # optional
        return int(psutil.virtual_memory().total)
    except Exception:
        pass
    try:
        return int(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES"))
    except (AttributeError, ValueError, OSError):
        return None


STARTED_UTC = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
_START_MONOTONIC = time.monotonic()
_gpu_rc, _gpu_out = (
    _capture(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
    if shutil.which("nvidia-smi")
    else (127, "nvidia-smi not found")
)
_freeze_rc, _freeze_out = _capture([sys.executable, "-m", "pip", "freeze"])
ENVIRONMENT = {
    "run_id": RUN_ID,
    "started_utc": STARTED_UTC,
    "python": sys.version,
    "executable": sys.executable,
    "python_version_requested": SPEC["python_version"],
    "platform": _platform.platform(),
    "machine": _platform.machine(),
    "processor": _platform.processor(),
    "cpu_count": os.cpu_count(),
    "ram_bytes": _ram_bytes(),
    "nvidia_smi": _gpu_out.strip(),
    "gpu_required": SPEC["gpu_required"],
    "pip_freeze": [line for line in _freeze_out.splitlines() if line.strip()],
    "notebook_digest": NOTEBOOK_DIGEST,
    "notebook_template_version": NOTEBOOK_TEMPLATE_VERSION,
    "spec_sha256": SPEC_SHA256,
}
(ARTIFACT_DIR / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2, sort_keys=True))
print(json.dumps({key: value for key, value in ENVIRONMENT.items() if key != "pip_freeze"}, indent=2, sort_keys=True))
print("workdir:", WORKDIR, "| export:", EXPORT_DIR)
if SPEC["gpu_required"] and _gpu_rc != 0:
    raise RuntimeError("gpu_required=True but nvidia-smi reported no usable device: refusing to continue")


In [ ]:
# Mandatory cell 2/8 — pinned environment.
# INSTALL_SPEC is the exact command recorded in the RunSpec; it must not silently resolve newer
# versions (it materialises the committed uv.lock). uv is bootstrapped first because the platforms
# do not ship it.
INSTALL_SPEC = SPEC["install_spec"]
if shutil.which("uv") is None:
    _require_ok(*_capture([sys.executable, "-m", "pip", "install", "-q", "uv"]), "uv bootstrap")

# The pinned lock lives in the repository, so the install cell needs the checkout at the pinned
# commit. Cell 3 re-checks out that commit and asserts HEAD == spec.git_commit.
if not (REPO_DIR / ".git").is_dir():
    _require_ok(*_capture(["git", "clone", SPEC["repo_url"], str(REPO_DIR)]), "git clone")
_require_ok(*_capture(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"]), "git fetch")
if SPEC["git_commit"]:
    _require_ok(*_capture(["git", "-C", str(REPO_DIR), "checkout", "--detach", SPEC["git_commit"]]), "git checkout")

print("install_spec:", INSTALL_SPEC)
_install_rc, _install_out = _capture(["bash", "-lc", INSTALL_SPEC], cwd=REPO_DIR)
(LOG_DIR / "install.log").write_text(_install_out)
_require_ok(_install_rc, _install_out, "pinned install")
_freeze_rc, _freeze_out = _capture([sys.executable, "-m", "pip", "freeze"])
(ARTIFACT_DIR / "dependencies.txt").write_text(_freeze_out)
print(_install_out[-2000:])


In [ ]:
# Mandatory cell 3/8 — repository at the exact commit.
# The assert below is the reproducibility gate: a run whose checkout differs from the recorded SHA
# is not evidence.
if not (REPO_DIR / ".git").is_dir():
    _require_ok(*_capture(["git", "clone", SPEC["repo_url"], str(REPO_DIR)]), "git clone")
_require_ok(*_capture(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"]), "git fetch")
if SPEC["git_commit"]:
    _require_ok(*_capture(["git", "-C", str(REPO_DIR), "checkout", "--detach", SPEC["git_commit"]]), "git checkout")
_head_rc, _head_out = _capture(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"])
HEAD_SHA = _head_out.strip()
if SPEC["git_commit"] and not SPEC["allow_dirty"]:
    assert HEAD_SHA == SPEC["git_commit"], (
        f"checkout mismatch: HEAD={HEAD_SHA} != spec.git_commit={SPEC['git_commit']}"
    )
elif SPEC["git_commit"] and HEAD_SHA != SPEC["git_commit"]:
    print(f"WARNING: allow_dirty=True and HEAD={HEAD_SHA} != spec.git_commit={SPEC['git_commit']}: this run is not reproducible")
sys.path.insert(0, str(REPO_DIR / "src"))
print("checked out", HEAD_SHA, "into", REPO_DIR)


In [ ]:
# Mandatory cell 4/8 — datasets, checksum-verified.
# Fetching lives in spectraquant.cloud.remote (repo code at the pinned commit), never inline here.
from spectraquant.cloud.remote import materialise_datasets

DATASET_RECORDS = materialise_datasets(
    config_path=SPEC["experiment_config"],
    overrides=SPEC["overrides"],
    dataset_refs=SPEC["dataset_refs"],
    dest_dir=WORKDIR / "data",
    seed=int(SPEC["seeds"][0]),
)
(ARTIFACT_DIR / "datasets.json").write_text(json.dumps(DATASET_RECORDS, indent=2, sort_keys=True))
print(json.dumps(DATASET_RECORDS, indent=2, sort_keys=True))


In [ ]:
# Mandatory cell 5/8 — the declared experiment, executed by repo code.
# `spectraquant run` is the frozen CLI entry point; the notebook adds no logic of its own. A non-zero
# exit is NOT raised here: the failure is recorded in the manifest (AGENTS.md §2b rule 4).
RUN_CONFIG = SPEC["experiment_config"]
RUN_OVERRIDES = list(SPEC["overrides"])
RUN_COMMAND = [sys.executable, "-m", "spectraquant", "run", "--config", RUN_CONFIG, *RUN_OVERRIDES]
RUN_LOG = LOG_DIR / "run.log"
RUN_RC, RUN_OUT = _capture(RUN_COMMAND, cwd=REPO_DIR)
RUN_LOG.write_text(RUN_OUT)
print("command:", " ".join(RUN_COMMAND))
print("exit:", RUN_RC)
print(RUN_OUT[-4000:])
if RUN_RC != 0 and "No such command" in RUN_OUT:
    print("HINT: this checkout has no `spectraquant run` subcommand; see scripts/cloud/README.md")


In [ ]:
# Mandatory cell 6/8 — run_manifest.json (validated against artifacts/schemas/ before writing).
import time as _time

from spectraquant.cloud.remote import build_run_manifest, collect_artifact_checksums, find_run_manifest, write_run_manifest

FINISHED_UTC = _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime())
WALL_TIME_S = round(_time.monotonic() - _START_MONOTONIC, 3)
# The run's own outputs (its manifest, checkpoints, tables) are part of the bundle.
if (REPO_DIR / "artifacts").is_dir():
    shutil.copytree(REPO_DIR / "artifacts", ARTIFACT_DIR / "artifacts", dirs_exist_ok=True)

_found_manifest = find_run_manifest(REPO_DIR, RUN_ID)
RUN_MANIFEST = json.loads(_found_manifest.read_text()) if _found_manifest else None
RUN_STATUS = "success" if RUN_RC == 0 else "failed"
FAILURE_REASON = None if RUN_RC == 0 else f"spectraquant run exited {RUN_RC}: {RUN_OUT[-500:].strip()}"
GPU_HOURS = round(WALL_TIME_S / 3600.0, 6) if SPEC["gpu_required"] else 0.0

MANIFEST_DOCUMENT = build_run_manifest(
    spec=SPEC,
    workdir=REPO_DIR,
    artifacts_dir=ARTIFACT_DIR,
    status=RUN_STATUS,
    started_utc=STARTED_UTC,
    finished_utc=FINISHED_UTC,
    environment=ENVIRONMENT,
    run_manifest=RUN_MANIFEST,
    failure_reason=FAILURE_REASON,
    log_path=RUN_LOG,
    remote_run_id=os.environ.get("SPECTRAQUANT_REMOTE_RUN_ID"),
    submitted_by=os.environ.get("SPECTRAQUANT_SUBMITTED_BY"),
    notebook_digest=NOTEBOOK_DIGEST,
    gpu_hours=GPU_HOURS,
    metrics={"cloud.wall_time_s": WALL_TIME_S},
)
CLOUD_CONTEXT = {
    "run_id": RUN_ID,
    "platform": SPEC["platform"],
    "spec_sha256": SPEC_SHA256,
    "notebook_digest": NOTEBOOK_DIGEST,
    "notebook_template_version": NOTEBOOK_TEMPLATE_VERSION,
    "git_commit": SPEC["git_commit"],
    "head_sha": HEAD_SHA,
    "status": RUN_STATUS,
    "started_utc": STARTED_UTC,
    "finished_utc": FINISHED_UTC,
    "wall_time_s": WALL_TIME_S,
    "gpu_hours": GPU_HOURS,
    "environment": ENVIRONMENT,
    "datasets": DATASET_RECORDS,
    "artifact_checksums": collect_artifact_checksums(ARTIFACT_DIR),
    "run_manifest_found": None if _found_manifest is None else str(_found_manifest),
    "install_spec": INSTALL_SPEC,
}
MANIFEST_PATH = write_run_manifest(out_dir=ARTIFACT_DIR, document=MANIFEST_DOCUMENT, cloud_context=CLOUD_CONTEXT)
print("manifest:", MANIFEST_PATH, "status:", RUN_STATUS)


In [ ]:
# Mandatory cell 7/8 — export the bundle and print the machine-readable result line.
import nbformat  # provided by the pinned environment


def _is_executed(path):
    """True when a notebook file carries execution counts (i.e. it actually ran)."""
    try:
        document = json.loads(Path(path).read_text())
    except (OSError, ValueError):
        return False
    cells = document.get("cells") or []
    return any(
        cell.get("cell_type") == "code" and cell.get("execution_count") is not None
        for cell in cells
        if isinstance(cell, dict)
    )


def _capture_executed_notebook():
    """Best-effort copy of the *executed* notebook into the export bundle."""
    destination = EXPORT_DIR / "notebook" / f"{RUN_ID}.ipynb"
    destination.parent.mkdir(parents=True, exist_ok=True)
    explicit = os.environ.get("SPECTRAQUANT_NOTEBOOK_PATH")
    if explicit and Path(explicit).is_file() and _is_executed(explicit):
        shutil.copy2(explicit, destination)
        return destination
    for candidate in sorted(Path.cwd().glob("*.ipynb")):
        if _is_executed(candidate):
            shutil.copy2(candidate, destination)
            return destination
    try:  # Colab: ask the front end for the live notebook (best effort, never fatal)
        from google.colab import _message
        payload = _message.blocking_request("get_ipynb", timeout_sec=30)["ipynb"]
        nbformat.write(nbformat.from_dict(payload), str(destination))
        return destination
    except Exception as exc:
        print(f"could not capture the executed notebook automatically: {type(exc).__name__}: {exc}")
    print(
        "ACTION REQUIRED: download the executed notebook from the platform into "
        f"{destination} — `spectraquant cloud collect` rejects a bundle without it."
    )
    return None


EXECUTED_NOTEBOOK = _capture_executed_notebook()
BUNDLE_FILES = {}
for _item in sorted(ARTIFACT_DIR.rglob("*")):
    if _item.is_file():
        _relative = _item.relative_to(ARTIFACT_DIR)
        _target = EXPORT_DIR / _relative
        _target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(_item, _target)
        BUNDLE_FILES[str(_relative)] = hashlib.sha256(_item.read_bytes()).hexdigest()
if EXECUTED_NOTEBOOK is not None:
    BUNDLE_FILES[str(EXECUTED_NOTEBOOK.relative_to(EXPORT_DIR))] = hashlib.sha256(
        EXECUTED_NOTEBOOK.read_bytes()
    ).hexdigest()

_drive = Path("/content/drive/MyDrive/spectraquant-runs")
if _drive.parent.is_dir():
    shutil.copytree(EXPORT_DIR, _drive / RUN_ID, dirs_exist_ok=True)
    print("mirrored to", _drive / RUN_ID)

RESULT = {
    "run_id": RUN_ID,
    "platform": SPEC["platform"],
    "status": RUN_STATUS,
    "export_dir": str(EXPORT_DIR),
    "manifest": str(EXPORT_DIR / "run_manifest.json"),
    "executed_notebook": None if EXECUTED_NOTEBOOK is None else str(EXECUTED_NOTEBOOK),
    "notebook_digest": NOTEBOOK_DIGEST,
    "spec_sha256": SPEC_SHA256,
    "git_commit": SPEC["git_commit"],
    "head_sha": HEAD_SHA,
    "gpu_hours": GPU_HOURS,
    "finished_utc": FINISHED_UTC,
    "files": BUNDLE_FILES,
}
print("SPECTRAQUANT_RESULT_JSON=" + json.dumps(RESULT, sort_keys=True, separators=(",", ":")))


In [ ]:
# Mandatory cell 8/8 — teardown marker: a truncated run is detectable because this line is missing.
TEARDOWN = {
    "run_id": RUN_ID,
    "status": RUN_STATUS,
    "teardown_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "manifest": str(MANIFEST_PATH),
}
print("SPECTRAQUANT_TEARDOWN_OK " + json.dumps(TEARDOWN, sort_keys=True))
